In [ ]:
import pandas as pd
import os
import re
import json # Library for handling JSON

# --- Configuration ---
# Define the filenames expected within the data folder
ORIGINAL_FILENAME = 'original.jsonl'
MODIFIED_FILENAME = 'modified.csv'

# --- Field Configuration for JSONL ---
# Field in the JSONL containing the text with ###QUESTION...###OPTIONS markers
JSONL_INPUT_FIELD = 'refined_arabic' # <<< *** VERIFY THIS FIELD NAME ***
# Fields in the JSONL containing the data to copy
JSONL_ABILITY_FIELD = 'ABILITY'
JSONL_INDEX_FIELD = 'INDEX' # Note: Case sensitive

# --- Hardcoded Paths ---
DATA_FOLDER_PATH = 'data'
OUTPUT_FILE_PATH = 'merged_output_jsonl_to_csv_question_match_v5.csv' # Changed output name
# Optional: Define path for mismatch report
MISMATCH_REPORT_PATH = 'mismatched_questions_report_v5.csv'

def extract_and_clean_question(text):
    """
    Extracts text between '###QUESTION' and '###OPTIONS' markers
    and cleans it aggressively. Handles variations in marker case/spacing.
    Returns the cleaned question text or empty string if markers not found/invalid or text is NaN.
    """
    if pd.isna(text):
        return ""
    text = str(text)
    # Use regex for flexible marker finding (case-insensitive, ignores extra spaces)
    # *** CORRECTED MARKER REGEX (Removed trailing ###) ***
    start_match = re.search(r'###\s*Question\b', text, re.IGNORECASE) # Look for ### Question (word boundary)
    end_match = re.search(r'###\s*Options\b', text, re.IGNORECASE)   # Look for ### Options (word boundary)

    if not start_match:
        # This specific input text does not contain the start marker
        return "" # Return empty if start marker not found

    start_pos = start_match.end() # Position *after* the start marker

    # Determine end position
    if end_match and end_match.start() > start_match.start():
        end_pos = end_match.start() # Position *before* the end marker
        question_text = text[start_pos : end_pos]
    else:
        # End marker not found after start marker, take rest of string
        # This might happen if OPTIONS marker is missing or malformed
        # print(f"Warning: '###OPTIONS' marker not found after '###QUESTION' in input text starting with: {text[:100]}...")
        question_text = text[start_pos:]

    # --- Aggressive Cleaning ---
    # Remove potential markdown code fences first
    question_text = re.sub(r'^```[a-zA-Z]*\s*', '', question_text)
    question_text = re.sub(r'\s*```$', '', question_text)
    # Replace different newline types and tabs with a space
    question_text = re.sub(r'[\r\n\t]+', ' ', question_text)
    # Replace non-breaking space (NBSP) with a regular space
    question_text = question_text.replace('\xa0', ' ')
    # Explicitly replace '---' marker sequence with a space, just in case
    question_text = question_text.replace('---', ' ')
    # Replace multiple whitespace characters with a single space
    question_text = re.sub(r'\s+', ' ', question_text)
    # Strip leading/trailing whitespace LAST
    question_text = question_text.strip()
    return question_text

def read_original_jsonl(jsonl_file_path):
    """
    Reads the JSONL file, extracts cleaned question and ABILITY/INDEX,
    and returns a lookup dictionary. Includes enhanced diagnostics.
    """
    lookup_dict = {}
    duplicates_count = 0
    errors_count = 0
    lines_processed_for_diag = 0
    max_diag_lines = 5 # Number of lines to print detailed diagnostics for

    print(f"Reading and processing original JSONL file: {jsonl_file_path}")
    try:
        with open(jsonl_file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                is_diag_line = (lines_processed_for_diag < max_diag_lines)
                if is_diag_line:
                    print(f"\n--- Processing JSONL Line {i+1} (Diagnostic) ---")

                try:
                    data = json.loads(line)
                    # Extract text from the specified field
                    input_text = data.get(JSONL_INPUT_FIELD)

                    if is_diag_line:
                        print(f"  Raw content of '{JSONL_INPUT_FIELD}': {str(input_text)[:200]}...") # Print first 200 chars

                    if not input_text:
                        if is_diag_line: print(f"  -> Skipping line: Field '{JSONL_INPUT_FIELD}' missing or empty.")
                        continue

                    cleaned_question = extract_and_clean_question(input_text)

                    if is_diag_line:
                        print(f"  Cleaned Question Extracted: '{cleaned_question[:200]}...'") # Print first 200 chars

                    if not cleaned_question:
                        if is_diag_line: print(f"  -> Skipping line: Could not extract/clean question.")
                        continue

                    ability = data.get(JSONL_ABILITY_FIELD)
                    index = data.get(JSONL_INDEX_FIELD)

                    if is_diag_line:
                         print(f"  Value for '{JSONL_ABILITY_FIELD}': {ability}")
                         print(f"  Value for '{JSONL_INDEX_FIELD}': {index}")

                    if ability is None or index is None:
                         if is_diag_line: print(f"  -> Skipping line: Missing '{JSONL_ABILITY_FIELD}' or '{JSONL_INDEX_FIELD}'.")
                         continue # Skip if essential data is missing

                    if cleaned_question in lookup_dict:
                        duplicates_count += 1
                        # Decide how to handle duplicates: overwrite (current) or keep first
                    lookup_dict[cleaned_question] = (ability, index)
                    if is_diag_line:
                        print(f"  -> Added/Updated question in lookup dictionary.")
                        lines_processed_for_diag += 1


                except json.JSONDecodeError:
                    print(f"Error: Could not decode JSON on line {i+1}")
                    errors_count += 1
                except KeyError as e:
                    print(f"Error: Missing key {e} on line {i+1}")
                    errors_count += 1
                except Exception as e:
                    print(f"Error processing line {i+1}: {e}")
                    errors_count += 1

        print(f"\nFinished processing JSONL. Found {len(lookup_dict)} unique questions.")
        if duplicates_count > 0:
            print(f"Warning: Encountered {duplicates_count} duplicate questions (last one seen was kept).")
        if errors_count > 0:
            print(f"Warning: Encountered {errors_count} errors during JSONL processing.")
        return lookup_dict

    except FileNotFoundError:
        print(f"Error: Original JSONL file not found at {jsonl_file_path}")
        return None
    except Exception as e:
        print(f"Error opening or reading JSONL file {jsonl_file_path}: {e}")
        return None


def merge_data(data_folder_path, output_file_path):
    """
    Main function to orchestrate reading files, merging, and saving.
    """
    original_file_path = os.path.join(data_folder_path, ORIGINAL_FILENAME)
    modified_file_path = os.path.join(data_folder_path, MODIFIED_FILENAME)

    print(f"Using hardcoded data folder: {data_folder_path}")
    print(f"Original data source (JSONL): {original_file_path}")
    print(f"Modified data source (CSV): {modified_file_path}")

    # 1. Read original JSONL into a lookup dictionary
    original_data_lookup = read_original_jsonl(original_file_path)
    if original_data_lookup is None:
        print("Stopping script due to error reading original file.")
        return

    # Check if lookup dictionary is empty after reading JSONL
    if not original_data_lookup:
        print("Error: Lookup dictionary is empty after processing JSONL. No questions were successfully extracted.")
        print("Please check the JSONL processing diagnostics above and verify the JSONL_INPUT_FIELD configuration and marker format.")
        # Optionally, you might still want to process the CSV and generate a mismatch report
        # return # Uncomment this to stop if no questions are found

    # 2. Read modified CSV
    try:
        df_modified = pd.read_csv(modified_file_path, encoding='utf-8')
        print(f"\nSuccessfully read modified CSV file: {modified_file_path}")
    except FileNotFoundError:
        print(f"Error: Modified CSV file not found at {modified_file_path}")
        return
    except Exception as e:
        print(f"Error reading modified CSV file {modified_file_path}: {e}")
        return

    # 3. Extract and clean questions from modified CSV
    # *** IMPORTANT: Assuming modified CSV uses [Question]...[Candidate Answers] ***
    # *** If modified CSV ALSO uses ###QUESTION###...###OPTIONS###, this needs adjustment ***
    print("Extracting and cleaning questions from modified CSV 'input' column...")
    # Temporarily define the CSV extraction function (might need adjustment)
    def extract_and_clean_question_csv(text):
        if pd.isna(text): return ""
        text = str(text)
        # Use regex for flexible marker finding (case-insensitive, ignores extra spaces)
        # *** Uses [...] markers for CSV ***
        start_match = re.search(r'\[\s*Question\s*\]', text, re.IGNORECASE)
        end_match = re.search(r'\[\s*Candidate Answers\s*\]', text, re.IGNORECASE)
        if not start_match: return ""
        start_pos = start_match.end()
        # Determine end position
        if end_match and end_match.start() > start_match.start():
            end_pos = end_match.start()
            question_text = text[start_pos : end_pos]
        else:
            question_text = text[start_pos:]
        # Cleaning steps
        question_text = re.sub(r'[\r\n\t]+', ' ', question_text)
        question_text = question_text.replace('\xa0', ' ')
        question_text = question_text.replace('---', ' ')
        question_text = re.sub(r'\s+', ' ', question_text)
        question_text = question_text.strip()
        return question_text

    df_modified['question_cleaned'] = df_modified['input'].apply(extract_and_clean_question_csv)
    print("Cleaning finished.")

    # --- CSV Diagnostics ---
    print("\n--- Running CSV Diagnostics ---")
    num_examples_to_check = 5
    print(f"Checking first {num_examples_to_check} cleaned questions from modified CSV against original JSONL data:")
    match_found_in_diag = False
    # Ensure the lookup dictionary exists before checking
    if original_data_lookup:
        for i, mod_question in enumerate(df_modified['question_cleaned'].head(num_examples_to_check)):
            if mod_question == "":
                 print(f"  Example {i+1}: Skipped (no question found/extracted in modified input).")
                 continue
            exists = mod_question in original_data_lookup
            print(f"  Example {i+1}: Exists in original JSONL keys? {exists}")
            if exists:
                match_found_in_diag = True
            # Optionally print the question text itself
            # print(f"    Modified Question (cleaned): '{mod_question[:100]}...'")
    else:
        print("  Skipping CSV diagnostics as the original data lookup dictionary is empty.")


    if match_found_in_diag:
         print(">>> Diagnostic check found at least one match! Merge might be successful. <<<")
    elif original_data_lookup: # Only print no match if lookup wasn't empty
         print(">>> Diagnostic check found no matches in the first examples. Merge might still fail. <<<")
    print("--- CSV Diagnostics Finished ---\n")

    # 4. Map data from lookup dictionary to modified DataFrame
    print("Mapping ABILITY and INDEX from original data to modified CSV...")

    # Ensure lookup dictionary exists before defining mapping functions
    if original_data_lookup:
        def get_ability(q):
            # Try matching the CSV question (extracted with [...]) against JSONL keys (extracted with ###...###)
            return original_data_lookup.get(q, (None, None))[0]

        def get_index(q):
            return original_data_lookup.get(q, (None, None))[1]

        # Apply the lookup functions
        df_modified[JSONL_ABILITY_FIELD] = df_modified['question_cleaned'].apply(get_ability)
        df_modified[JSONL_INDEX_FIELD] = df_modified['question_cleaned'].apply(get_index)
    else:
        # If lookup is empty, create empty columns
        print("  Original data lookup is empty, creating empty ABILITY/INDEX columns.")
        df_modified[JSONL_ABILITY_FIELD] = None
        df_modified[JSONL_INDEX_FIELD] = None


    # 5. Verification
    print("Mapping complete.")
    total_rows = len(df_modified)
    missing_ability = df_modified[JSONL_ABILITY_FIELD].isnull().sum()
    missing_index = df_modified[JSONL_INDEX_FIELD].isnull().sum() # Should be same as ability
    successful_merges = total_rows - missing_ability

    print(f"Number of rows in final merged file: {total_rows}")
    print(f"Number of rows successfully merged (found match): {successful_merges}")
    print(f"Number of rows with missing '{JSONL_ABILITY_FIELD}'/'{JSONL_INDEX_FIELD}' (no match found): {missing_ability}")

    # 6. Create Mismatch Report (Optional)
    # Generate report even if lookup was empty, showing all rows failed
    if missing_ability > 0 and MISMATCH_REPORT_PATH:
        print(f"Generating mismatch report...")
        # Select rows where the merge failed
        mismatched_df = df_modified[df_modified[JSONL_ABILITY_FIELD].isnull()]
        # Keep only the original input and the cleaned question for the report
        mismatched_df_report = mismatched_df[['input', 'question_cleaned']].copy()
        try:
            # Construct full path for the report
            mismatch_output_path = os.path.join(os.path.dirname(output_file_path) or '.', MISMATCH_REPORT_PATH)
            mismatched_df_report.to_csv(mismatch_output_path, index=False)
            print(f"Mismatch report saved to: {mismatch_output_path}")
        except Exception as e:
            print(f"Error saving mismatch report: {e}")


    # 7. Clean up temporary column
    df_merged = df_modified.drop(columns=['question_cleaned'])
    print("Removed temporary 'question_cleaned' column.")

    # 8. Save Output
    try:
        df_merged.to_csv(output_file_path, index=False)
        print(f"Successfully saved merged data to: {output_file_path}")
    except Exception as e:
        print(f"Error saving merged file: {e}")


# --- Script Execution ---
if __name__ == "__main__":
    print("--- Starting Merge Process (JSONL to CSV) ---")
    merge_data(DATA_FOLDER_PATH, OUTPUT_FILE_PATH)
    print("--- Merge Process Finished ---")


--- Starting Merge Process (JSONL to CSV) ---
Using hardcoded data folder: data
Original data source (JSONL): data/original.jsonl
Modified data source (CSV): data/modified.csv
Reading and processing original JSONL file: data/original.jsonl

--- Processing JSONL Line 1 (Diagnostic) ---
  Raw content of 'refined_arabic': ```arabic
###STORY
في أمسية عطلة شتوية، يلعب وانغ لي، وليو تينغ، وشياو مينغ بالألعاب معًا في المنزل. يلعبون أولاً بسيارة كهربائية، ثم يلعبون بلعبة الأحجية. في هذا الوقت، يقول وانغ لي، "لديّ ما أفعله و...
  Cleaned Question Extracted: 'ماذا سيفعل شياو مينغ على الأرجح؟...'
  Value for 'ABILITY': Knowledge: Knowledge-attention links
  Value for 'INDEX': 1
  -> Added/Updated question in lookup dictionary.

--- Processing JSONL Line 2 (Diagnostic) ---
  Raw content of 'refined_arabic': ```arabic
###STORY
في ظهيرة يوم مشمس في نهاية الأسبوع، تلعب لي جوان ووانغ تشيانغ وشياو فانغ معًا في المنزل. يبدأون بلعبة مسلية أولى، وهي لغز متاهة ثلاثي الأبعاد، ثم ينتقلون إلى لعبة ممتعة أخرى،